In [ ]:
import numpy as np
import pandas as pd
import os
import ast
import pdb
from openai._client import OpenAI
df = pd.read_csv('data/test_llama_formatted.csv')
df.head()
from openai._client import OpenAI
client = OpenAI(api_key='xxxxxxxxxxx') # OpenAI api key

def embed(texts):
    embeddings = client.embeddings.create(
        input=texts,
        model='text-embedding-ada-002'
    )
    return [x.embedding for x in embeddings.data]
def get_embedding(text, model = 'text-embedding-ada-002'):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding
get_embedding(df['data'].iloc[0])
df['data'].head(3).apply(get_embedding)
df['embeddings'] = df['data'].apply(get_embedding)
df.to_csv('data/train_llama_with_embeddings.csv', index=False)
df.to_pickle('data/train_llama_with_embeddings.pkl')
df = pd.read_pickle('data/train_llama_with_embeddings.pkl')
df.head()
question = 'Write SOAP based on the given data.'
question_embedding = get_embedding(question)
question, question_embedding[0:10], "..."
def fn(page_embedding):
  return np.dot(page_embedding, question_embedding)
df['distance'] = df['embeddings'].apply(fn)
df.head()
df.sort_values('distance', ascending=False, inplace=True)
df.head()
context = df['data'].iloc[0] + '\n' + df['data'].iloc[1] + '\n' + df['data'].iloc[2]
context
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "system", "content": "You are an assistant who is helping to write a SOAP using doctor-patient conversation"},
              {"role": "user", "content": question},
              {"role": "assistant", "content": f"Use this information from Doctor Patient Query to answer the user question: {context}. Please stick to the question while answering."}],
)
response.choices[0].message.content
def query(question):
  question_embedding = get_embedding(question)

  def fn(page_embedding):
    return np.dot(page_embedding, question_embedding)

  distance_series = df['embeddings'].apply(fn)
  top_four = distance_series.sort_values(ascending=False).index[0:4]

  text_series= df.loc[top_four]['data']
  context = '\n\n'.join(text_series)

  response = client.chat.completions.create(
      model="gpt-3.5-turbo",
      messages=[{"role": "system", "content": "You are an assistant who is helping to write a SOAP using doctor-patient conversation"},
                {"role": "user", "content": question},
                {"role": "assistant", "content": f"Use this information from Doctor Patient Query to answer the user question: {context}. Please stick to the question while answering."}])
  return response.choices[0].message.content
query("Do you have any medical conditions?")
